In [ ]:
%reload_ext autoreload
%autoreload 2

from tqdm import trange
from flygym.compose import ActuatorType

import importlib
import miniproject.simulation
importlib.reload(miniproject.simulation)

from miniproject.simulation import MiniprojectSimulation
from submission.controller import Controller
from submission.controller import Controller, add_state_overlay

import mediapy
#import cv2
import matplotlib.pyplot as plt

sim = MiniprojectSimulation(level=2, seed=1)
print(sim.enable_wind)  # should print True
controller = Controller(sim)

actual_wind_angles = []

from flygym.compose import ActuatorType
import numpy as np


for _ in trange(50000): # 50 000 steps pour atteindre la cible sur flat
    joint_angles, adhesion = controller.step(sim)
    sim.set_actuator_inputs(sim.fly.name, ActuatorType.POSITION, joint_angles)
    sim.set_actuator_inputs(sim.fly.name, ActuatorType.ADHESION, adhesion)
    sim.step()
    sim.render_as_needed()
    actual_wind_angles.append(getattr(sim, 'current_wind_angle', None))


n_frames = len(sim.renderer.frames["birdeyecam"])
n_steps = len(controller.trajectory_states)
step_ratio = n_steps // n_frames

annotated = add_state_overlay(
    sim.renderer.frames["birdeyecam"],
    controller.trajectory_states,
    step_ratio,
    actual_wind_angles=actual_wind_angles,
    perceived_wind_angles=controller.trajectory_wind_perceived,
    headings=controller.trajectory_heading
)


mediapy.show_video(annotated, fps=sim.renderer.output_fps, title="top-down view with state")


mediapy.show_video(controller.frames, fps=sim.renderer.output_fps, title="ommatidia vision")
sim.renderer.show_in_notebook()

controller.plot_trajectory("ma_trajectoire.png")
controller.plot_trajectory_with_states("trajectory_states.png")


Failed to read module file 'C:\Users\estel\AppData\Roaming\uv\python\cpython-3.12-windows-x86_64-none\Lib\urllib\parse.py' for module 'urllib.parse': UnicodeDecodeError
Traceback (most recent call last):
  File "c:\Users\estel\Documents\mouche\cobar-2026\Controlling_BAR_Project\.venv\Lib\site-packages\IPython\core\extensions.py", line 62, in load_extension
    return self._load_extension(module_str)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\estel\Documents\mouche\cobar-2026\Controlling_BAR_Project\.venv\Lib\site-packages\IPython\core\extensions.py", line 77, in _load_extension
    mod = import_module(module_str)
          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\estel\AppData\Roaming\uv\python\cpython-3.12-windows-x86_64-none\Lib\importlib\__init__.py", line 90, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 1387, in _gcd_im

False


 78%|███████▊  | 38874/50000 [00:47<00:12, 920.45it/s]c:\Users\estel\Documents\mouche\cobar-2026\Controlling_BAR_Project\miniproject\submission\controller.py:1033: RuntimeWarning: divide by zero encountered in scalar divide
  self.pitch_derivative += ((self.prev_pitch - self.pitch)/self.speed)
 78%|███████▊  | 38967/50000 [00:47<00:12, 907.15it/s]c:\Users\estel\Documents\mouche\cobar-2026\Controlling_BAR_Project\miniproject\submission\controller.py:1033: RuntimeWarning: invalid value encountered in scalar add
  self.pitch_derivative += ((self.prev_pitch - self.pitch)/self.speed)
 78%|███████▊  | 39153/50000 [00:47<00:11, 913.79it/s]

mean_odor is :  1.4221434915187233e-05


100%|██████████| 50000/50000 [00:59<00:00, 838.76it/s]


top-down view with state This browser does not support the video tag.


ommatidia vision This browser does not support the video tag.


nmf/backcam This browser does not support the video tag.


birdeyecam This browser does not support the video tag.


📊 Trajectoire sauvegardée : ma_trajectoire.png
   - Points de trajectoire : 50000


In [ ]:
print("mean value of left eye: ", controller.left_intensity)
print("mean value of right eye: ", controller.right_intensity)

AttributeError: 'Controller' object has no attribute 'left_intensity'

In [ ]:
image = controller.frames[-18]  
plt.imshow(image)

In [ ]:
heights = [290, 300, 310]

fig, axes = plt.subplots(8, 1, figsize=(12, 20))

for idx, h in enumerate(heights):
    line = image[h, :, :]  # tous les pixels à la hauteur h, shape (900, 3)

    axes[idx].plot(line[:, 0], color='red',   label='R')
    axes[idx].plot(line[:, 1], color='green', label='G')
    axes[idx].plot(line[:, 2], color='blue',  label='B')

    axes[idx].set_title(f'Hauteur {h}')
    axes[idx].set_xlabel('Position horizontale')
    axes[idx].set_ylabel('Intensité')
    axes[idx].legend()

plt.tight_layout()
plt.show()